# 🇮🇩 Indonesian Lip Reading Dataset — Data Collection & Preprocessing

**Tujuan / Purpose:**  
Pipeline lengkap untuk mengumpulkan dan memproses data video bibir untuk proyek *lip reading* Bahasa Indonesia.

> ⚠️ **Kompatibilitas:** Notebook ini menggunakan **MediaPipe Tasks API** (`mediapipe >= 0.10`)  
> API lama (`mp.solutions.face_mesh`) sudah dihapus di versi terbaru — notebook ini **100% kompatibel dengan Python 3.11.9**.

---

## 📋 Pipeline Overview

```
1. Setup & Installation
2. Download Model (.task file)  → FaceLandmarker model dari Google
3. Video Data Collection        → Rekam video dari webcam (2-4 detik, ~10 FPS)
4. Frame Extraction             → Ekstrak frame dari setiap video
5. Lip Region Extraction        → Crop area bibir menggunakan MediaPipe Tasks FaceLandmarker
6. Dataset Structure            → Organisasi folder + labels.csv
7. Data Validation              → Periksa kualitas dan kelengkapan dataset
8. Visualization                → Preview frame dan animasi
```

---
**Constraints:**
- Durasi rekaman: 2-4 detik
- Target FPS: ~10 FPS → ~20-40 frame per sampel
- Ukuran frame bibir: 112x112 piksel
- **Tidak ada pelatihan model** — hanya pengumpulan & preprocessing data

**Tech Stack:** Python 3.11.9 · OpenCV · MediaPipe >= 0.10 (Tasks API) · NumPy · Matplotlib · IPython


---
## Cell 1 — Install Dependencies

In [13]:
# =============================================================================
# CELL 1: Install required libraries
# Jalankan sekali, restart kernel jika diminta.
# =============================================================================

import subprocess, sys

def install(package: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

required = [
    "opencv-python",
    "mediapipe>=0.10.0",   # Tasks API — mp.solutions sudah dihapus di versi ini
    "numpy",
    "matplotlib",
    "pandas",
    "Pillow",
    "ipywidgets",
    "tqdm",
    "requests",            # Untuk download model .task
]

for pkg in required:
    print(f"Installing {pkg}...")
    install(pkg)

import mediapipe as mp
print(f"\nSemua dependensi berhasil diinstall!")
print(f"   MediaPipe version: {mp.__version__}")
print(f"   Python version   : {sys.version.split()[0]}")

Installing opencv-python...



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Installing mediapipe>=0.10.0...



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Installing numpy...


...


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Installing tqdm...



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Installing requests...

Semua dependensi berhasil diinstall!
   MediaPipe version: 0.10.5
   Python version   : 3.11.9



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## Cell 2 — Imports & Global Configuration

In [14]:
# =============================================================================
# CELL 2: Imports dan konfigurasi global
# Edit bagian CONFIG untuk menyesuaikan pipeline.
# =============================================================================

import cv2
import numpy as np
import pandas as pd
import os
import time
import shutil
import requests
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, HTML, clear_output
from PIL import Image
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# MediaPipe Tasks API — Cara import yang benar untuk mediapipe >= 0.10
#
# LAMA (dihapus, JANGAN dipakai):
#   import mediapipe as mp
#   mp.solutions.face_mesh.FaceMesh()
#
# BARU (Tasks API, digunakan di sini):
#   from mediapipe.tasks.python.vision import FaceLandmarker, FaceLandmarkerOptions
# =============================================================================
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import (
    FaceLandmarker,
    FaceLandmarkerOptions,
    RunningMode,
)

print(f"MediaPipe {mp.__version__} — Tasks API loaded")

# =============================================================================
# GLOBAL CONFIG — Edit sesuai kebutuhan
# =============================================================================

CONFIG = {
    # Recording
    "CAMERA_INDEX"          : 0,           # 0 = webcam default; coba 1/2 untuk kamera eksternal
    "RECORD_FPS"            : 30,          # FPS hardware kamera
    "TARGET_FPS"            : 10,          # FPS target setelah downsampling
    "MIN_DURATION_SEC"      : 2.0,         # Durasi minimum klip
    "MAX_DURATION_SEC"      : 4.0,         # Durasi maksimum klip
    "COUNTDOWN_SEC"         : 3,           # Hitung mundur sebelum rekam

    # Lip crop
    "LIP_FRAME_SIZE"        : (112, 112),  # Ukuran output crop bibir (W, H)
    "LIP_PADDING"           : 0.30,        # Padding di sekitar bounding box bibir

    # Paths
    "DATASET_ROOT"          : "dataset",   # Folder root dataset
    "MODEL_DIR"             : "models",    # Folder untuk file model .task

    # Validation
    "MIN_FRAMES_PER_SAMPLE" : 15,
    "MAX_FRAMES_PER_SAMPLE" : 60,
}

# Derived paths
DATASET_ROOT  = Path(CONFIG["DATASET_ROOT"])
RAW_VIDEO_DIR = DATASET_ROOT / "raw_videos"
PROCESSED_DIR = DATASET_ROOT / "processed_frames"
LABELS_CSV    = DATASET_ROOT / "labels.csv"
MODEL_DIR     = Path(CONFIG["MODEL_DIR"])
MODEL_PATH    = MODEL_DIR / "face_landmarker.task"

for d in [RAW_VIDEO_DIR, PROCESSED_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dataset root  : {DATASET_ROOT.resolve()}")
print(f"Target FPS    : {CONFIG['TARGET_FPS']}")
print(f"Durasi klip   : {CONFIG['MIN_DURATION_SEC']}-{CONFIG['MAX_DURATION_SEC']}s")
print(f"Ukuran crop   : {CONFIG['LIP_FRAME_SIZE']}")

MediaPipe 0.10.5 — Tasks API loaded
Dataset root  : /home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/building-model/dataset
Target FPS    : 10
Durasi klip   : 2.0-4.0s
Ukuran crop   : (112, 112)


---
## Cell 3 — Download FaceLandmarker Model

MediaPipe Tasks API membutuhkan file model `.task` yang harus didownload terpisah.  
File ini hanya perlu didownload **satu kali** (~3.5 MB).

In [15]:
# =============================================================================
# CELL 3: Download face_landmarker.task model
#
# Mengapa perlu file ini?
#   Berbeda dari API lama yang modelnya langsung ada di package,
#   MediaPipe Tasks API memisahkan model sebagai file .task.
#   File ini berisi graph definisi + bobot model.
# =============================================================================

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

def download_model(url: str, dest: Path, force: bool = False) -> None:
    """Download model file jika belum ada."""
    if dest.exists() and not force:
        size_mb = dest.stat().st_size / (1024 * 1024)
        print(f"Model sudah ada: {dest.name} ({size_mb:.1f} MB) — skip download.")
        return

    print(f"Downloading {dest.name} dari Google Storage...")
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(
        desc=dest.name,
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

    print(f"Model berhasil didownload: {dest}")


def make_face_landmarker(running_mode=RunningMode.IMAGE) -> FaceLandmarker:
    """
    Factory helper — buat instance FaceLandmarker dengan opsi standar.

    Parameters
    ----------
    running_mode : RunningMode.IMAGE   → untuk gambar statis (frame individual)
                   RunningMode.VIDEO   → untuk frame berurutan dengan timestamp
                   RunningMode.LIVE_STREAM → async, untuk stream real-time
    """
    base_opts = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
    opts = FaceLandmarkerOptions(
        base_options                          = base_opts,
        running_mode                          = running_mode,
        num_faces                             = 1,
        min_face_detection_confidence         = 0.5,
        min_face_presence_score               = 0.5,
        min_tracking_confidence               = 0.5,
        output_face_blendshapes               = False,
        output_facial_transformation_matrixes = False,
    )
    return FaceLandmarker.create_from_options(opts)


# Download model
download_model(MODEL_URL, MODEL_PATH)

# Verifikasi model valid
try:
    _test = make_face_landmarker()
    _test.close()
    print("FaceLandmarker model valid dan siap digunakan!")
except Exception as e:
    print(f"Model error: {e}")
    print("Coba jalankan ulang cell ini dengan force=True di download_model()")

Model sudah ada: face_landmarker.task (3.6 MB) — skip download.
Model error: FaceLandmarkerOptions.__init__() got an unexpected keyword argument 'min_face_presence_score'
Coba jalankan ulang cell ini dengan force=True di download_model()


---
## Cell 4 — Sample ID & Label Manager

In [16]:
# =============================================================================
# CELL 4: Sample ID manager + labels.csv helper
# =============================================================================

def get_next_sample_id() -> str:
    """Return next sample_XXX ID berdasarkan labels.csv yang sudah ada."""
    if LABELS_CSV.exists():
        df = pd.read_csv(LABELS_CSV)
        if not df.empty:
            last_num = df["sample_id"].str.extract(r"(\d+)").astype(int).max().values[0]
            return f"sample_{last_num + 1:03d}"
    return "sample_001"


def append_label(sample_id: str, transcription: str, speaker_id: str) -> None:
    """Tambahkan satu baris ke labels.csv."""
    row = {
        "sample_id"     : sample_id,
        "transcription" : transcription.strip(),
        "speaker_id"    : speaker_id.strip(),
        "recorded_at"   : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    if LABELS_CSV.exists():
        df = pd.read_csv(LABELS_CSV)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(LABELS_CSV, index=False)


def load_labels() -> pd.DataFrame:
    """Load labels.csv atau kembalikan DataFrame kosong."""
    if LABELS_CSV.exists():
        return pd.read_csv(LABELS_CSV)
    return pd.DataFrame(columns=["sample_id", "transcription", "speaker_id", "recorded_at"])


def get_video_path(sample_id: str, label: str, speaker_id: str, index: int) -> Path:
    """
    Naming convention: <label>_<speakerID>_<index>.mp4
    Contoh: apa_spk01_001.mp4
    """
    safe_label = label.replace(" ", "_").lower()
    return RAW_VIDEO_DIR / f"{safe_label}_{speaker_id}_{index:03d}.mp4"


print("Label manager siap.")
print(f"   Sample ID berikutnya: {get_next_sample_id()}")

Label manager siap.
   Sample ID berikutnya: sample_002


---
## Cell 5 — Video Recording dengan Countdown & Live Preview

**Cara pakai:**
1. Jalankan cell ini — fungsi `record_clip()` akan terdefinisi
2. Panggil dari Cell 6 (sesi interaktif) atau manual
3. Tekan **`s`** untuk simpan & berhenti lebih awal, **`q`** untuk batalkan

> Pastikan jendela OpenCV aktif/fokus agar tombol keyboard berfungsi.

In [19]:
def draw_lip_landmarks_on_frame(
    frame: np.ndarray,
    face_landmarks_list,
    h: int,
    w: int
) -> None:
    """
    Gambar titik landmark bibir di atas frame dengan visualisasi enhanced:
    - Garis penghubung outer lip (hijau)
    - Garis penghubung inner lip (orange)
    - Titik landmark besar dengan ring
    - Bounding box bibir

    face_landmarks_list: List[NormalizedLandmark]
        — hasil dari result.face_landmarks[0] pada Tasks API
    """
    if not face_landmarks_list:
        return

    # Konversi ke piksel
    points = []
    for idx in ALL_LIP_IDX:
        if idx < len(face_landmarks_list):
            lm = face_landmarks_list[idx]
            x, y = int(lm.x * w), int(lm.y * h)
            points.append((x, y))

    if not points:
        return

    # === 1. Draw outer lip outline ===
    outer_pts = [points[i] for i in range(min(len(LIP_OUTER), len(points)))]
    if len(outer_pts) > 2:
        pts_arr = np.array(outer_pts, dtype=np.int32)
        cv2.polylines(frame, [pts_arr], isClosed=True, 
                      color=(100, 255, 100), thickness=2, lineType=cv2.LINE_AA)

    # === 2. Draw inner lip outline ===
    inner_pts = [points[len(LIP_OUTER) + i] for i in range(len(LIP_INNER))
                 if (len(LIP_OUTER) + i) < len(points)]
    if len(inner_pts) > 2:
        pts_arr = np.array(inner_pts, dtype=np.int32)
        cv2.polylines(frame, [pts_arr], isClosed=True,
                      color=(255, 150, 50), thickness=2, lineType=cv2.LINE_AA)

    # === 3. Draw landmark points (besar & terang) ===
    for x, y in points:
        # Background circle
        cv2.circle(frame, (x, y), 4, (50, 50, 50), -1, cv2.LINE_AA)
        # Main circle (terang hijau)
        cv2.circle(frame, (x, y), 3, (0, 255, 100), -1, cv2.LINE_AA)
        # Ring
        cv2.circle(frame, (x, y), 4, (0, 255, 50), 1, cv2.LINE_AA)

    # === 4. Draw bounding box bibir ===
    xs = [face_landmarks_list[i].x * w for i in ALL_LIP_IDX
          if i < len(face_landmarks_list)]
    ys = [face_landmarks_list[i].y * h for i in ALL_LIP_IDX
          if i < len(face_landmarks_list)]
    
    if xs and ys:
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        pad_x = (x_max - x_min) * 0.30
        pad_y = (y_max - y_min) * 0.30
        
        x1 = int(max(0, x_min - pad_x))
        y1 = int(max(0, y_min - pad_y))
        x2 = int(min(w, x_max + pad_x))
        y2 = int(min(h, y_max + pad_y))
        
        # Kotak utama
        cv2.rectangle(frame, (x1, y1), (x2, y2), (100, 200, 255), 2, cv2.LINE_AA)
        # Sudut-sudut
        corner_len = 10
        for px, py in [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]:
            cv2.circle(frame, (px, py), 3, (100, 200, 255), -1, cv2.LINE_AA)

---
## Cell 6 — Sesi Rekaman Interaktif

Edit `SPEAKER_ID` dan `WORDS_TO_RECORD`, lalu jalankan cell ini.

In [ ]:
# =============================================================================
# CELL 6: Sesi rekaman interaktif
# Edit SPEAKER_ID dan WORDS_TO_RECORD sebelum menjalankan
# =============================================================================

SPEAKER_ID = "spk01"         # Ganti untuk setiap pembicara: spk01, spk02, ...

WORDS_TO_RECORD = [
    # Vokal Bahasa Indonesia
    "a", "i", "u", "e", "o",
    # Kata-kata umum
    "apa", "iya", "tidak", "makan", "minum",
    "pergi", "pulang", "terima kasih", "halo", "selamat",
    # Tambahkan kata lain di sini...
]

REPETITIONS_PER_WORD = 3     # Jumlah rekaman per kata
SHOW_LANDMARKS       = True   # ✓ TRUE = tampilkan landmark bibir saat rekam (BAGUS UNTUK QUALITY CHECK!)

# =============================================================================

print("=" * 60)
print(" INDONESIAN LIP READING — SESI REKAMAN")
print("=" * 60)
print(f"  Pembicara      : {SPEAKER_ID}")
print(f"  Jumlah kata    : {len(WORDS_TO_RECORD)} label")
print(f"  Repetisi       : {REPETITIONS_PER_WORD}x per kata")
print(f"  Total klip     : {len(WORDS_TO_RECORD) * REPETITIONS_PER_WORD}")
print(f"  ✓ Landmark     : AKTIF (lihat garis & titik hijau/oranye)")
print("=" * 60)

total_recorded = 0
total_skipped  = 0

for word in WORDS_TO_RECORD:
    print(f"\n--- Label: '{word}' ---")
    for rep in range(1, REPETITIONS_PER_WORD + 1):
        print(f"  Rep {rep}/{REPETITIONS_PER_WORD} — Enter untuk mulai, 'skip' untuk lewati.")
        user_input = input("  > ").strip().lower()
        if user_input == "skip":
            print("  Dilewati.")
            total_skipped += 1
            continue

        result = record_clip(
            label          = word,
            speaker_id     = SPEAKER_ID,
            rep_index      = rep,
            show_landmarks = SHOW_LANDMARKS,  # ✓ Sekarang TRUE!
        )
        if result:
            total_recorded += 1
        else:
            total_skipped += 1

print("\n" + "=" * 60)
print(f"  Sesi selesai! Direkam: {total_recorded}  |  Dilewati: {total_skipped}")
print(f"  Video disimpan di: {RAW_VIDEO_DIR}")
print("=" * 60)

 INDONESIAN LIP READING — SESI REKAMAN
  Pembicara      : spk01
  Jumlah kata    : 15 label
  Repetisi       : 3x per kata
  Total klip     : 45
  ✓ Landmark     : AKTIF (lihat garis & titik hijau/oranye)

--- Label: 'a' ---
  Rep 1/3 — Enter untuk mulai, 'skip' untuk lewati.
Kamera tidak bisa dibuka. Cek CAMERA_INDEX di CONFIG.
  Rep 2/3 — Enter untuk mulai, 'skip' untuk lewati.


[ WARN:0@517.417] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x5643648c0940] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@517.420] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


Kamera tidak bisa dibuka. Cek CAMERA_INDEX di CONFIG.
  Rep 3/3 — Enter untuk mulai, 'skip' untuk lewati.


[ WARN:0@520.064] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x5643648c0940] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@520.066] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


Kamera tidak bisa dibuka. Cek CAMERA_INDEX di CONFIG.

--- Label: 'i' ---
  Rep 1/3 — Enter untuk mulai, 'skip' untuk lewati.


...

Kamera tidak bisa dibuka. Cek CAMERA_INDEX di CONFIG.
  Rep 2/3 — Enter untuk mulai, 'skip' untuk lewati.


[ WARN:0@524.056] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x5643648c0940] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@524.058] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


Kamera tidak bisa dibuka. Cek CAMERA_INDEX di CONFIG.
  Rep 3/3 — Enter untuk mulai, 'skip' untuk lewati.


[ WARN:0@524.153] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x5643648c0940] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@524.153] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


Kamera tidak bisa dibuka. Cek CAMERA_INDEX di CONFIG.

--- Label: 'o' ---
  Rep 1/3 — Enter untuk mulai, 'skip' untuk lewati.


---
## Cell 7 — Ekstraksi Frame (~10 FPS)

In [ ]:
# =============================================================================
# CELL 7: Ekstraksi frame — downsample ke TARGET_FPS
#
# Logika downsampling:
#   frame_step = round(source_fps / target_fps)
#   Contoh: kamera 30 FPS, target 10 FPS -> ambil setiap frame ke-3
# =============================================================================

def extract_frames_from_video(
    video_path : Path,
    sample_id  : str,
    target_fps : int  = CONFIG["TARGET_FPS"],
    overwrite  : bool = False
) -> int:
    """
    Ekstrak frame dari video pada target_fps dan simpan sebagai JPEG.
    Kembalikan jumlah frame yang disimpan, atau -1 jika gagal.
    """
    out_dir = PROCESSED_DIR / sample_id

    if out_dir.exists() and not overwrite:
        existing = len(list(out_dir.glob("*.jpg")))
        if existing > 0:
            return existing  # Sudah diproses

    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  Tidak bisa membuka: {video_path.name}")
        return -1

    source_fps = cap.get(cv2.CAP_PROP_FPS) or CONFIG["RECORD_FPS"]
    frame_step = max(1, round(source_fps / target_fps))

    frame_idx   = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_step == 0:
            fname = out_dir / f"frame_{saved_count:04d}.jpg"
            cv2.imwrite(str(fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved_count += 1
        frame_idx += 1

    cap.release()
    return saved_count


def extract_all_frames(overwrite: bool = False) -> dict:
    """Proses semua video di raw_videos/ berdasarkan urutan di labels.csv."""
    labels_df = load_labels()
    if labels_df.empty:
        print("labels.csv kosong. Rekam beberapa klip terlebih dahulu.")
        return {}

    videos = sorted(RAW_VIDEO_DIR.glob("*.mp4"))
    if not videos:
        print(f"Tidak ada video di {RAW_VIDEO_DIR}")
        return {}

    print(f"Mengekstrak frame dari {len(videos)} video pada {CONFIG['TARGET_FPS']} FPS...")
    summary = {}

    for i, video_path in enumerate(tqdm(videos, desc="Ekstraksi")):
        sample_id = labels_df.iloc[i]["sample_id"] if i < len(labels_df) else f"sample_{i+1:03d}"
        n_frames  = extract_frames_from_video(video_path, sample_id, overwrite=overwrite)
        summary[sample_id] = n_frames

    ok  = {k: v for k, v in summary.items() if v > 0}
    err = {k: v for k, v in summary.items() if v <= 0}

    print(f"\nSelesai. {len(ok)} sampel diekstrak, {len(err)} gagal.")
    if ok:
        print(f"Jumlah frame: min={min(ok.values())}, max={max(ok.values())}")
    return summary


# Jalankan ekstraksi
frame_summary = extract_all_frames(overwrite=False)

if frame_summary:
    print("\nRingkasan frame per sampel:")
    for sid, n in frame_summary.items():
        lo, hi = CONFIG["MIN_FRAMES_PER_SAMPLE"], CONFIG["MAX_FRAMES_PER_SAMPLE"]
        icon   = "OK" if lo <= n <= hi else "PERIKSA"
        print(f"  [{icon}] {sid}: {n} frame")

---
## Cell 8 — Ekstraksi Lip Region (MediaPipe Tasks API)

Ini adalah **langkah preprocessing inti**. Perbedaan utama dari API lama:

| | API Lama (dihapus) | API Baru (digunakan di sini) |
|---|---|---|
| Import | `mp.solutions.face_mesh` | `from mediapipe.tasks.python.vision import FaceLandmarker` |
| Init | `FaceMesh(static_image_mode=True)` | `FaceLandmarker.create_from_options(opts)` |
| Proses | `face_mesh.process(rgb)` | `landmarker.detect(mp.Image(...))` |
| Hasil | `result.multi_face_landmarks` | `result.face_landmarks` |
| Landmark | `landmark.x`, `.y` | sama, `NormalizedLandmark.x`, `.y` |

In [ ]:
# =============================================================================
# CELL 8a: Utilitas visualisasi lip landmark — Tasks API compatible
#
# DITINGKATKAN:
#   - Landmark lebih besar dan lebih terang
#   - Garis penghubung antar landmark (outline + inner)
#   - Bounding box bibir yang jelas
#   - Statistik confidence & jarak antar titik
# =============================================================================

def get_lip_bbox(
    face_landmarks_list,   # List[NormalizedLandmark] dari result.face_landmarks[0]
    img_h: int,
    img_w: int,
    padding: float = CONFIG["LIP_PADDING"]
):
    """
    Hitung padded bounding box dari landmark bibir.

    Parameters
    ----------
    face_landmarks_list : List[NormalizedLandmark]
        Dari result.face_landmarks[0] — wajah pertama yang terdeteksi.
        NormalizedLandmark memiliki atribut .x, .y, .z (nilai 0.0-1.0)
    img_h, img_w : Dimensi gambar asli dalam piksel
    padding      : Fraksi padding di sekitar bibir (default 30%)

    Returns (x1, y1, x2, y2) dalam piksel, clamped ke batas gambar.
    Returns None jika landmark kosong.
    """
    if not face_landmarks_list:
        return None

    # Konversi koordinat normalized ke piksel
    xs = [face_landmarks_list[i].x * img_w for i in ALL_LIP_IDX
          if i < len(face_landmarks_list)]
    ys = [face_landmarks_list[i].y * img_h for i in ALL_LIP_IDX
          if i < len(face_landmarks_list)]

    if not xs or not ys:
        return None

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    # Padding proporsional terhadap ukuran bibir
    pad_x = (x_max - x_min) * padding
    pad_y = (y_max - y_min) * padding

    x1 = int(max(0,     x_min - pad_x))
    y1 = int(max(0,     y_min - pad_y))
    x2 = int(min(img_w, x_max + pad_x))
    y2 = int(min(img_h, y_max + pad_y))

    return (x1, y1, x2, y2)


def draw_lip_landmarks_on_frame_enhanced(
    frame: np.ndarray,
    face_landmarks_list,
    h: int,
    w: int,
    show_stats: bool = True
) -> dict:
    """
    Gambar lip landmarks dengan enhanced visualization:
    - Titik besar berwarna terang
    - Garis penghubung outer & inner
    - Bounding box bibir
    - Statistik confidence & jarak

    Returns dict dengan statistik landmark.
    """
    if not face_landmarks_list:
        return {"detected": False}

    # Konversi ke piksel
    points = []
    confidences = []
    for idx in ALL_LIP_IDX:
        if idx < len(face_landmarks_list):
            lm = face_landmarks_list[idx]
            x, y = int(lm.x * w), int(lm.y * h)
            points.append((x, y))
            confidences.append(lm.presence if hasattr(lm, 'presence') else 0.99)

    if not points:
        return {"detected": False}

    # === 1. Draw outer lip outline (garis penghubung outer) ===
    outer_pts = [points[i] for i in range(len(LIP_OUTER)) 
                 if i < len(points)]
    if len(outer_pts) > 2:
        # Buat polygon (garis tertutup)
        pts_arr = np.array(outer_pts, dtype=np.int32)
        cv2.polylines(frame, [pts_arr], isClosed=True, 
                      color=(100, 255, 100), thickness=2, lineType=cv2.LINE_AA)

    # === 2. Draw inner lip outline (garis penghubung inner) ===
    inner_pts = [points[len(LIP_OUTER) + i] for i in range(len(LIP_INNER))
                 if (len(LIP_OUTER) + i) < len(points)]
    if len(inner_pts) > 2:
        pts_arr = np.array(inner_pts, dtype=np.int32)
        cv2.polylines(frame, [pts_arr], isClosed=True,
                      color=(255, 150, 50), thickness=2, lineType=cv2.LINE_AA)

    # === 3. Draw landmark points (besar & jelas) ===
    for i, (x, y) in enumerate(points):
        # Background circle (lebih besar)
        cv2.circle(frame, (x, y), 4, (50, 50, 50), -1, cv2.LINE_AA)
        # Main circle (terang)
        cv2.circle(frame, (x, y), 3, (0, 255, 100), -1, cv2.LINE_AA)
        # Outer ring untuk confidence
        conf = confidences[i] if i < len(confidences) else 0.9
        ring_color = (0, 255, 50) if conf > 0.7 else (0, 150, 200)
        cv2.circle(frame, (x, y), 4, ring_color, 1, cv2.LINE_AA)

    # === 4. Draw bounding box ===
    bbox = get_lip_bbox(face_landmarks_list, h, w)
    if bbox:
        x1, y1, x2, y2 = bbox
        # Kotak utama
        cv2.rectangle(frame, (x1, y1), (x2, y2), (100, 200, 255), 2, cv2.LINE_AA)
        # Sudut-sudut (highlight)
        corner_len = 15
        for px, py in [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]:
            cv2.circle(frame, (px, py), 3, (100, 200, 255), -1, cv2.LINE_AA)

    # === 5. Hitung statistik ===
    mean_conf = np.mean(confidences) if confidences else 0
    
    # Hitung jarak antar titik (untuk smoothness check)
    distances = []
    for i in range(len(points) - 1):
        dx = points[i+1][0] - points[i][0]
        dy = points[i+1][1] - points[i][1]
        dist = np.sqrt(dx**2 + dy**2)
        distances.append(dist)
    
    max_dist = max(distances) if distances else 0
    mean_dist = np.mean(distances) if distances else 0

    stats = {
        "detected": True,
        "num_points": len(points),
        "mean_confidence": mean_conf,
        "max_distance": max_dist,
        "mean_distance": mean_dist,
        "bbox": bbox,
    }

    return stats


def stabilise_bbox_sequence(bboxes: list, smooth_window: int = 5) -> list:
    """
    Haluskan bounding box antar frame dengan sliding-window average.
    Ini mengurangi jitter crop antar frame yang berurutan.

    bboxes       : list of (x1, y1, x2, y2) atau None
    smooth_window: ukuran jendela rata-rata (default 5 frame)

    Returns list bbox yang dihaluskan dengan panjang sama dengan input.
    """
    # Forward fill: isi None dengan nilai valid terdekat
    filled, last_valid = [], None
    for b in bboxes:
        if b is not None:
            last_valid = b
        filled.append(last_valid)

    # Backward fill untuk None di awal
    first_valid = next((b for b in filled if b is not None), None)
    if first_valid is None:
        return bboxes  # Semua frame gagal deteksi
    filled = [first_valid if b is None else b for b in filled]

    arr      = np.array(filled, dtype=float)   # shape: (N, 4)
    smoothed = np.zeros_like(arr)
    half     = smooth_window // 2

    for i in range(len(arr)):
        lo = max(0, i - half)
        hi = min(len(arr), i + half + 1)
        smoothed[i] = arr[lo:hi].mean(axis=0)

    return [tuple(int(v) for v in row) for row in smoothed]


print("Utilitas lip landmark (Tasks API) — ENHANCED — siap.")
print("  ✓ Landmark visualization dengan garis & bounding box")
print("  ✓ Statistik confidence & jarak antar titik")


In [ ]:

# =============================================================================
# CELL 8b: Enhanced Lip crop + Landmark Annotation
#
# Fitur BARU:
#   1. Simpan frame annotated (dengan landmark terlihat) → annotated_frames/
#   2. Simpan frame cropped (final 112x112) → processed_frames/
#   3. Simpan statistik landmark per frame → landmark_stats/
#
# Alur kerja 2-pass:
#   Pass 1 → FaceLandmarker IMAGE mode → kumpulkan bbox + anotasi
#   Pass 2 → Stabilisasi bbox → crop + resize + simpan
# =============================================================================

# Setup directory untuk output baru
ANNOTATED_DIR = DATASET_ROOT / "annotated_frames"   # Frame dengan landmark visible
STATS_DIR     = DATASET_ROOT / "landmark_stats"     # CSV statistik per sampel
for d in [ANNOTATED_DIR, STATS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directories:")
print(f"  Annotated  : {ANNOTATED_DIR}")
print(f"  Stats      : {STATS_DIR}")


def crop_lips_for_sample_enhanced(
    sample_id   : str,
    output_size : tuple = CONFIG["LIP_FRAME_SIZE"],
    overwrite   : bool  = False,
    save_annotated : bool = True
) -> dict:
    """
    Crop dan resize area bibir untuk semua frame dalam satu sampel.
    PLUS: Simpan frame annotated dengan landmark terlihat.

    Returns dict: {total_frames, detected, failed, success_rate, stats_df}
    """
    sample_dir = PROCESSED_DIR / sample_id
    frames     = sorted(sample_dir.glob("frame_*.jpg"))

    if not frames:
        return {
            "total_frames": 0, "detected": 0, "failed": 0,
            "success_rate": 0.0, "stats_df": None
        }

    # Cek apakah sudah diproses
    if not overwrite:
        test_img = cv2.imread(str(frames[0]))
        if test_img is not None and test_img.shape[:2] == (output_size[1], output_size[0]):
            return {
                "total_frames": len(frames), "detected": len(frames),
                "failed": 0, "success_rate": 1.0, "stats_df": None
            }

    # Setup direktori untuk output
    annotated_sample_dir = ANNOTATED_DIR / sample_id
    if save_annotated:
        annotated_sample_dir.mkdir(parents=True, exist_ok=True)

    # Buat FaceLandmarker IMAGE mode untuk batch processing
    landmarker = make_face_landmarker(RunningMode.IMAGE)

    # PASS 1: Deteksi landmark + hitung bbox untuk semua frame + anotasi
    raw_images = []
    raw_bboxes = []
    landmark_stats_list = []

    for frame_idx, frame_path in enumerate(frames):
        img = cv2.imread(str(frame_path))
        if img is None:
            raw_images.append(None)
            raw_bboxes.append(None)
            landmark_stats_list.append({"frame_id": frame_idx, "detected": False})
            continue

        h, w = img.shape[:2]

        # Konversi ke RGB
        rgb      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        # Jalankan deteksi
        result = landmarker.detect(mp_image)

        if result.face_landmarks:
            bbox = get_lip_bbox(result.face_landmarks[0], h, w)
            # Get enhanced visualization + statistics
            stats = draw_lip_landmarks_on_frame_enhanced(
                img, result.face_landmarks[0], h, w, show_stats=True
            )
        else:
            bbox = None
            stats = {"detected": False}

        raw_images.append(img)
        raw_bboxes.append(bbox)
        
        # Simpan statistik per frame
        stats_record = {
            "frame_id": frame_idx,
            "detected": stats.get("detected", False),
            "confidence": stats.get("mean_confidence", 0.0),
            "max_distance": stats.get("max_distance", 0.0),
            "mean_distance": stats.get("mean_distance", 0.0),
        }
        landmark_stats_list.append(stats_record)

    landmarker.close()

    # PASS 2: Stabilisasi bbox lalu crop + resize + simpan
    stable_bboxes = stabilise_bbox_sequence(raw_bboxes)
    detected = 0
    failed   = 0

    for img, bbox, frame_path in zip(raw_images, stable_bboxes, frames):
        if img is None or bbox is None:
            failed += 1
            continue

        x1, y1, x2, y2 = bbox
        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            failed += 1
            continue

        # Resize ke output_size
        crop_resized = cv2.resize(crop, output_size, interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(str(frame_path), crop_resized, [cv2.IMWRITE_JPEG_QUALITY, 95])
        detected += 1

    # PASS 3: Simpan frame annotated (jika diaktifkan)
    if save_annotated and raw_images and raw_bboxes:
        for img, bbox, frame_path in zip(raw_images, stable_bboxes, frames):
            if img is not None and bbox is not None:
                # Draw pada copy agar original tidak terganggu
                img_annotated = img.copy()
                x1, y1, x2, y2 = bbox
                # Bounding box
                cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (100, 200, 255), 2)
                # Frame number text
                cv2.putText(img_annotated, f"{frame_path.stem}", (15, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                # Simpan
                out_path = annotated_sample_dir / frame_path.name
                cv2.imwrite(str(out_path), img_annotated, [cv2.IMWRITE_JPEG_QUALITY, 90])

    # Convert stats ke DataFrame dan simpan
    stats_df = pd.DataFrame(landmark_stats_list)
    stats_csv_path = STATS_DIR / f"{sample_id}.csv"
    stats_df.to_csv(stats_csv_path, index=False)

    total        = len(frames)
    success_rate = detected / total if total > 0 else 0.0

    return {
        "total_frames" : total,
        "detected"     : detected,
        "failed"       : failed,
        "success_rate" : success_rate,
        "stats_df"     : stats_df,
    }


def crop_lips_all_samples_enhanced(overwrite: bool = False) -> pd.DataFrame:
    """Jalankan lip cropping enhanced untuk semua direktori sampel."""
    sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
    if not sample_dirs:
        print("Tidak ada direktori sampel. Jalankan ekstraksi frame terlebih dahulu.")
        return pd.DataFrame()

    print(f"Mengekstrak region bibir + anotasi dari {len(sample_dirs)} sampel...")
    records = []

    for sd in tqdm(sample_dirs, desc="Crop bibir + Annotate"):
        report = crop_lips_for_sample_enhanced(sd.name, overwrite=overwrite)
        report_summary = {
            "sample_id": sd.name,
            "total_frames": report["total_frames"],
            "detected": report["detected"],
            "failed": report["failed"],
            "success_rate": report["success_rate"],
        }
        records.append(report_summary)

    df  = pd.DataFrame(records)
    avg = df["success_rate"].mean()

    print(f"\nLip extraction enhanced selesai.")
    print(f"  Rata-rata detection rate    : {avg:.1%}")
    print(f"  Sampel dengan <80% deteksi  : {(df['success_rate'] < 0.8).sum()}")
    print(f"  ✓ Annotated frames disimpan : {ANNOTATED_DIR}")
    print(f"  ✓ Landmark stats disimpan   : {STATS_DIR}")
    return df


# Jalankan lip cropping enhanced
print("Menjalankan lip extraction dengan landmark annotation...\n")
lip_report = crop_lips_all_samples_enhanced(overwrite=False)

if not lip_report.empty:
    print("\nLaporan per sampel:")
    display(lip_report.to_string(index=False))


---
## Cell 9 — Landmark Statistics & Preview Annotated Frames

Lihat statistik confidence landmark dan preview frame dengan landmark terlihat.


In [ ]:
# =============================================================================
# CELL 9: Landmark Statistics & Annotated Frame Preview
# =============================================================================

def load_landmark_stats(sample_id: str) -> pd.DataFrame:
    """Load landmark statistics CSV untuk satu sampel."""
    stats_csv = STATS_DIR / f"{sample_id}.csv"
    if stats_csv.exists():
        return pd.read_csv(stats_csv)
    return pd.DataFrame()


def preview_annotated_sample(sample_id: str, max_frames: int = 12) -> None:
    """Preview frame dengan landmark terlihat."""
    annotated_dir = ANNOTATED_DIR / sample_id
    frames        = sorted(annotated_dir.glob("frame_*.jpg"))

    if not frames:
        print(f"Tidak ada annotated frame untuk {sample_id}")
        return

    label = "???"
    df    = load_labels()
    if not df.empty and sample_id in df["sample_id"].values:
        label = df[df["sample_id"] == sample_id]["transcription"].values[0]

    indices  = np.linspace(0, len(frames) - 1, min(max_frames, len(frames)), dtype=int)
    selected = [frames[i] for i in indices]

    ncols     = min(6, len(selected))
    nrows     = (len(selected) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5 + 0.5))
    fig.suptitle(f"Annotated: {sample_id} | Label: '{label}' | {len(frames)} frame",
                 fontsize=12, fontweight="bold")

    axes_flat = np.array(axes).flatten()
    for i, fp in enumerate(selected):
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        axes_flat[i].imshow(img)
        axes_flat[i].set_title(f"#{indices[i]:03d}", fontsize=8)
        axes_flat[i].axis("off")
    for j in range(len(selected), len(axes_flat)):
        axes_flat[j].axis("off")

    plt.tight_layout()
    plt.show()


def show_landmark_stats_summary() -> None:
    """Tampilkan ringkasan statistik landmark untuk semua sampel."""
    sample_ids = sorted([
        d.name for d in PROCESSED_DIR.iterdir() if d.is_dir()
    ])

    if not sample_ids:
        print("Tidak ada sampel.")
        return

    print(f"\nLandmark Detection Statistics ({len(sample_ids)} sampel)\n")
    print(f"{'Sample ID':<15} {'Det.Rate':>10} {'Conf.Avg':>10} {'Max.Dist':>10} {'Status':>8}")
    print("-" * 55)

    for sid in sample_ids:
        stats_df = load_landmark_stats(sid)
        if stats_df.empty:
            print(f"{sid:<15} {'N/A':>10} {'N/A':>10} {'N/A':>10} {'❌':>8}")
            continue

        detected = stats_df[stats_df["detected"] == True]
        if len(detected) == 0:
            print(f"{sid:<15} {'0%':>10} {'-':>10} {'-':>10} {'❌':>8}")
        else:
            det_rate = len(detected) / len(stats_df) * 100
            conf_avg = detected["confidence"].mean()
            max_dist = detected["max_distance"].max()

            status = "✓ OK" if det_rate >= 80 else "⚠ LOW"
            print(f"{sid:<15} {det_rate:>9.0f}% {conf_avg:>10.3f} {max_dist:>10.1f} {status:>8}")


# Tampilkan ringkasan
show_landmark_stats_summary()

print("\n" + "="*55)
print("Untuk preview frame dengan landmark, jalankan:")
print("  preview_annotated_sample('sample_001')")
print("="*55)


In [ ]:
# =============================================================================
# CELL 9: Tampilkan struktur dataset dan labels.csv
# =============================================================================

def print_dataset_tree(max_samples: int = 5) -> None:
    sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
    n_videos    = len(list(RAW_VIDEO_DIR.glob("*.mp4")))
    labels_size = f"{LABELS_CSV.stat().st_size} bytes" if LABELS_CSV.exists() else "(belum ada)"

    print(f"dataset/")
    print(f"  +-- raw_videos/           ({n_videos} video)")
    print(f"  +-- processed_frames/     ({len(sample_dirs)} sampel)")
    for i, sd in enumerate(sample_dirs[:max_samples]):
        n    = len(list(sd.glob("*.jpg")))
        conn = "+--" if i < len(sample_dirs[:max_samples]) - 1 else "L--"
        print(f"  |   {conn} {sd.name}/   ({n} frame)")
    if len(sample_dirs) > max_samples:
        print(f"  |   L-- ... ({len(sample_dirs) - max_samples} sampel lainnya)")
    print(f"  L-- labels.csv            ({labels_size})")


print_dataset_tree()
print()

labels_df = load_labels()
if not labels_df.empty:
    print(f"labels.csv — {len(labels_df)} entri:")
    display(labels_df)
else:
    print("labels.csv kosong atau belum ada.")

---
## Cell 10 — Validasi Data & Quality Control

In [ ]:
# =============================================================================
# CELL 10: Validasi dataset secara menyeluruh
#
# Pemeriksaan:
#   [1] sample_id ada di labels.csv
#   [2] Jumlah frame dalam rentang MIN-MAX
#   [3] Semua frame berukuran 112x112
#   [4] Tidak ada frame corrupt/tidak bisa dibaca
# =============================================================================

def validate_dataset() -> pd.DataFrame:
    labels_df   = load_labels()
    sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])

    if not sample_dirs:
        print("Tidak ada sampel yang sudah diproses.")
        return pd.DataFrame()

    EXPECTED_W, EXPECTED_H = CONFIG["LIP_FRAME_SIZE"]
    MIN_F = CONFIG["MIN_FRAMES_PER_SAMPLE"]
    MAX_F = CONFIG["MAX_FRAMES_PER_SAMPLE"]

    rows = []
    print(f"Memvalidasi {len(sample_dirs)} sampel...\n")

    for sd in tqdm(sample_dirs, desc="Validasi"):
        sid      = sd.name
        frames   = sorted(sd.glob("frame_*.jpg"))
        n_frames = len(frames)
        issues   = []

        # Cek 1: Ada di labels.csv?
        if labels_df.empty or sid not in labels_df["sample_id"].values:
            issues.append("TIDAK ada di labels.csv")
            transcription = "???"
        else:
            transcription = labels_df[labels_df["sample_id"] == sid]["transcription"].values[0]

        # Cek 2: Jumlah frame
        if n_frames < MIN_F:
            issues.append(f"Frame terlalu sedikit ({n_frames} < {MIN_F})")
        elif n_frames > MAX_F:
            issues.append(f"Frame terlalu banyak ({n_frames} > {MAX_F})")

        # Cek 3: Integritas dan ukuran
        corrupt  = 0
        bad_size = 0
        for fp in frames:
            img = cv2.imread(str(fp))
            if img is None:
                corrupt += 1
                continue
            h, w = img.shape[:2]
            if (w, h) != (EXPECTED_W, EXPECTED_H):
                bad_size += 1

        if corrupt  > 0: issues.append(f"{corrupt} frame corrupt")
        if bad_size > 0: issues.append(f"{bad_size} frame ukuran salah")

        rows.append({
            "sample_id"    : sid,
            "transcription": transcription,
            "n_frames"     : n_frames,
            "corrupt"      : corrupt,
            "wrong_size"   : bad_size,
            "issues"       : "; ".join(issues) if issues else "OK",
            "status"       : "PASS" if not issues else "FAIL",
        })

    df    = pd.DataFrame(rows)
    n_ok  = (df["status"] == "PASS").sum()
    n_bad = len(df) - n_ok

    print(f"\nRingkasan Validasi")
    print(f"  Total sampel    : {len(df)}")
    print(f"  Lulus           : {n_ok}")
    print(f"  Bermasalah      : {n_bad}")
    if len(df) > 0:
        print(f"  Rata-rata frame : {df['n_frames'].mean():.1f}")
        print(f"  Min frame       : {df['n_frames'].min()}")
        print(f"  Max frame       : {df['n_frames'].max()}")
    return df


validation_report = validate_dataset()

if not validation_report.empty:
    print("\nLaporan validasi lengkap:")
    display(validation_report)

---
## Cell 11 — Statistik Dataset

In [ ]:
# =============================================================================
# CELL 11: Plot statistik dataset
# =============================================================================

def plot_dataset_stats() -> None:
    labels_df   = load_labels()
    sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])

    if not sample_dirs or labels_df.empty:
        print("Tidak ada data untuk diplot.")
        return

    frame_counts = {sd.name: len(list(sd.glob("*.jpg"))) for sd in sample_dirs}
    fc_df = pd.DataFrame.from_dict(frame_counts, orient="index", columns=["frames"])
    fc_df.index.name = "sample_id"
    fc_df = fc_df.merge(
        labels_df[["sample_id", "transcription"]],
        left_index=True, right_on="sample_id", how="left"
    )
    label_counts = labels_df["transcription"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Indonesian Lip Reading Dataset — Statistik", fontsize=14, fontweight="bold")

    # Plot 1: Jumlah frame per sampel
    ax1    = axes[0]
    colors = [
        "#2ecc71" if CONFIG["MIN_FRAMES_PER_SAMPLE"] <= v <= CONFIG["MAX_FRAMES_PER_SAMPLE"]
        else "#e74c3c"
        for v in fc_df["frames"]
    ]
    ax1.bar(range(len(fc_df)), fc_df["frames"], color=colors)
    ax1.axhline(CONFIG["MIN_FRAMES_PER_SAMPLE"], color="orange", ls="--",
                label=f"Min ({CONFIG['MIN_FRAMES_PER_SAMPLE']})")
    ax1.axhline(CONFIG["MAX_FRAMES_PER_SAMPLE"], color="red",    ls="--",
                label=f"Max ({CONFIG['MAX_FRAMES_PER_SAMPLE']})")
    ax1.set_title("Jumlah Frame per Sampel")
    ax1.set_xlabel("Indeks Sampel")
    ax1.set_ylabel("# Frame")
    ax1.legend(fontsize=8)
    ax1.set_xticks(range(len(fc_df)))
    ax1.set_xticklabels(fc_df["sample_id"], rotation=45, ha="right", fontsize=6)

    # Plot 2: Jumlah sampel per label
    ax2 = axes[1]
    label_counts.plot(kind="bar", ax=ax2, color="#3498db")
    ax2.set_title("Sampel per Label")
    ax2.set_xlabel("Label")
    ax2.set_ylabel("Jumlah")
    ax2.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    out_path = DATASET_ROOT / "dataset_stats.png"
    plt.savefig(str(out_path), dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Chart disimpan ke {out_path}")


plot_dataset_stats()

---
## Cell 12 — Preview Frame (Grid Statis)

In [ ]:
# =============================================================================
# CELL 12: Preview grid frame bibir untuk satu sampel
# =============================================================================

def preview_sample(sample_id: str, max_frames: int = 12) -> None:
    sample_dir = PROCESSED_DIR / sample_id
    frames     = sorted(sample_dir.glob("frame_*.jpg"))

    if not frames:
        print(f"Tidak ada frame untuk {sample_id}")
        return

    label = "???"
    df    = load_labels()
    if not df.empty and sample_id in df["sample_id"].values:
        label = df[df["sample_id"] == sample_id]["transcription"].values[0]

    indices  = np.linspace(0, len(frames) - 1, min(max_frames, len(frames)), dtype=int)
    selected = [frames[i] for i in indices]

    ncols     = min(6, len(selected))
    nrows     = (len(selected) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2 + 0.5))
    fig.suptitle(f"Sampel: {sample_id}  |  Label: '{label}'  |  {len(frames)} frame",
                 fontsize=12, fontweight="bold")

    axes_flat = np.array(axes).flatten()
    for i, fp in enumerate(selected):
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        axes_flat[i].imshow(img)
        axes_flat[i].set_title(f"f{indices[i]:03d}", fontsize=7)
        axes_flat[i].axis("off")
    for j in range(len(selected), len(axes_flat)):
        axes_flat[j].axis("off")

    plt.tight_layout()
    plt.show()


sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
if sample_dirs:
    preview_sample(sample_dirs[0].name)
else:
    print("Belum ada sampel yang diproses.")

---
## Cell 13 — Animasi Sekuens Bibir

In [ ]:
# =============================================================================
# CELL 13: Animasi sekuens bibir dalam notebook
# =============================================================================

def animate_sample(sample_id: str, fps: int = CONFIG["TARGET_FPS"]) -> None:
    sample_dir = PROCESSED_DIR / sample_id
    frames     = sorted(sample_dir.glob("frame_*.jpg"))

    if not frames:
        print(f"Tidak ada frame untuk {sample_id}")
        return

    imgs  = [cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
             for fp in frames if cv2.imread(str(fp)) is not None]

    label = "???"
    df    = load_labels()
    if not df.empty and sample_id in df["sample_id"].values:
        label = df[df["sample_id"] == sample_id]["transcription"].values[0]

    fig, ax = plt.subplots(figsize=(3, 3.5))
    ax.axis("off")
    fig.suptitle(f"'{label}'  ({len(imgs)} frame @ {fps} FPS)", fontsize=10)
    im         = ax.imshow(imgs[0])
    frame_text = ax.set_title(f"Frame 0001/{len(imgs):04d}", fontsize=8)

    def update(i):
        im.set_data(imgs[i])
        frame_text.set_text(f"Frame {i+1:04d}/{len(imgs):04d}")
        return [im, frame_text]

    ani = animation.FuncAnimation(
        fig, update, frames=len(imgs),
        interval=int(1000 / fps), blit=True, repeat=True
    )
    plt.tight_layout()
    display(HTML(ani.to_jshtml()))
    plt.close(fig)


sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
if sample_dirs:
    animate_sample(sample_dirs[0].name)
else:
    print("Belum ada sampel yang diproses.")

---
## Cell 14 — Inspeksi Sampel Interaktif

In [ ]:
# =============================================================================
# CELL 14: Widget inspeksi sampel interaktif (ipywidgets)
# =============================================================================

sample_dirs = sorted([d.name for d in PROCESSED_DIR.iterdir() if d.is_dir()])

if not sample_dirs:
    print("Belum ada sampel. Selesaikan pipeline terlebih dahulu.")
else:
    sample_dd = widgets.Dropdown(
        options     = sample_dirs,
        description = "Sampel:",
        layout      = widgets.Layout(width="200px")
    )
    mode_dd = widgets.Dropdown(
        options     = [("Grid statis", "grid"), ("Animasi", "anim")],
        description = "Mode:",
        layout      = widgets.Layout(width="200px")
    )
    btn = widgets.Button(description="Tampilkan", button_style="primary")
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output(wait=True)
            if mode_dd.value == "grid":
                preview_sample(sample_dd.value)
            else:
                animate_sample(sample_dd.value)

    btn.on_click(on_click)
    display(widgets.HBox([sample_dd, mode_dd, btn]))
    display(out)

---
## Cell 15 — Hapus Sampel Buruk

In [ ]:
# =============================================================================
# CELL 15: Hapus sampel bermasalah
# DRY_RUN = True  -> hanya tampilkan, tidak hapus
# DRY_RUN = False -> benar-benar hapus dari disk dan labels.csv
# =============================================================================

DRY_RUN = True  # Ganti ke False untuk benar-benar menghapus

def remove_bad_samples(dry_run: bool = True) -> list:
    labels_df   = load_labels()
    sample_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
    bad_samples = []

    for sd in sample_dirs:
        sid      = sd.name
        frames   = sorted(sd.glob("frame_*.jpg"))
        n_frames = len(frames)
        reason   = []

        if n_frames < CONFIG["MIN_FRAMES_PER_SAMPLE"]:
            reason.append(f"frame terlalu sedikit ({n_frames})")

        corrupt = sum(1 for fp in frames if cv2.imread(str(fp)) is None)
        if corrupt == n_frames and n_frames > 0:
            reason.append("semua frame corrupt")

        if reason:
            bad_samples.append((sid, "; ".join(reason)))

    if not bad_samples:
        print("Tidak ada sampel buruk. Dataset bersih!")
        return []

    prefix = "[DRY RUN] " if dry_run else ""
    print(f"{prefix}Ditemukan {len(bad_samples)} sampel bermasalah:")
    for sid, r in bad_samples:
        print(f"  FAIL  {sid}: {r}")

    if not dry_run:
        removed = []
        for sid, _ in bad_samples:
            shutil.rmtree(str(PROCESSED_DIR / sid), ignore_errors=True)
            if not labels_df.empty:
                labels_df = labels_df[labels_df["sample_id"] != sid]
            removed.append(sid)
            print(f"  DELETED: {sid}")
        labels_df.to_csv(LABELS_CSV, index=False)
        print(f"\n{len(removed)} sampel dihapus. labels.csv diperbarui.")
        return removed
    else:
        print("\nSet DRY_RUN = False untuk benar-benar menghapus sampel ini.")
        return [s[0] for s in bad_samples]


_ = remove_bad_samples(dry_run=DRY_RUN)

---
## Cell 16 — Export Laporan Dataset

In [ ]:
# =============================================================================
# CELL 16: Export laporan dataset ke Markdown
# =============================================================================

def export_summary_report() -> None:
    labels_df    = load_labels()
    sample_dirs  = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
    total_frames = sum(len(list(d.glob("*.jpg"))) for d in sample_dirs)

    lines = [
        "# Indonesian Lip Reading Dataset — Laporan Ringkasan",
        f"\nDibuat pada: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "\n---\n",
        "## Environment",
        f"- Python          : {__import__('sys').version.split()[0]}",
        f"- MediaPipe       : {mp.__version__} (Tasks API — mp.solutions TIDAK digunakan)",
        "\n---\n",
        "## Statistik Dataset",
        f"- Total sampel    : {len(sample_dirs)}",
        f"- Total frame     : {total_frames}",
        f"- Label unik      : {labels_df['transcription'].nunique() if not labels_df.empty else 0}",
        f"- Pembicara unik  : {labels_df['speaker_id'].nunique() if not labels_df.empty else 0}",
        f"- Target FPS      : {CONFIG['TARGET_FPS']}",
        f"- Ukuran frame    : {CONFIG['LIP_FRAME_SIZE'][0]}x{CONFIG['LIP_FRAME_SIZE'][1]} px",
        "\n---\n",
        "## Konfigurasi",
        f"- Durasi min      : {CONFIG['MIN_DURATION_SEC']}s",
        f"- Durasi max      : {CONFIG['MAX_DURATION_SEC']}s",
        f"- Padding bibir   : {CONFIG['LIP_PADDING']:.0%}",
        f"- Frame min/sampel: {CONFIG['MIN_FRAMES_PER_SAMPLE']}",
        f"- Frame max/sampel: {CONFIG['MAX_FRAMES_PER_SAMPLE']}",
        "\n---\n",
        "## Distribusi Label",
    ]

    if not labels_df.empty:
        lines.append("| Transkripsi | Jumlah |")
        lines.append("|---|---|")
        for label, count in labels_df["transcription"].value_counts().items():
            lines.append(f"| {label} | {count} |")

    report_path = DATASET_ROOT / "REPORT.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Laporan disimpan ke {report_path}")


export_summary_report()

print()
print("=" * 60)
print(" Pipeline Selesai!")
print("=" * 60)
print()
print(" Langkah selanjutnya:")
print("  1. Rekam lebih banyak pembicara (ganti SPEAKER_ID di Cell 6)")
print("  2. Tambahkan kata ke WORDS_TO_RECORD")
print("  3. Jalankan ulang Cell 7-16 untuk memproses rekaman baru")
print("  4. Gunakan labels.csv + processed_frames/ untuk training model")
print()

---

## Quick Reference

| Cell | Tujuan | Fungsi Utama |
|------|--------|-------------|
| 1 | Install paket | `pip install mediapipe>=0.10` |
| 2 | Config & imports | `CONFIG` dict |
| 3 | **Download model `.task`** | `download_model()` + `make_face_landmarker()` |
| 4 | ID + label manager | `get_next_sample_id()`, `append_label()` |
| 5 | Fungsi rekaman inti | `record_clip()` (Tasks VIDEO mode) |
| 6 | **Sesi rekaman interaktif** | Edit `WORDS_TO_RECORD` |
| 7 | Ekstraksi frame | `extract_all_frames()` |
| 8a | Utilitas lip bbox | `get_lip_bbox()`, `stabilise_bbox_sequence()` |
| 8b | **Crop bibir semua sampel** | `crop_lips_all_samples()` (Tasks IMAGE mode) |
| 9 | Struktur dataset | `print_dataset_tree()` |
| 10 | **Validasi data** | `validate_dataset()` |
| 11 | Plot statistik | `plot_dataset_stats()` |
| 12 | Grid frame statis | `preview_sample()` |
| 13 | Animasi sekuens | `animate_sample()` |
| 14 | Inspeksi interaktif | ipywidgets dropdown |
| 15 | Hapus sampel buruk | `remove_bad_samples()` |
| 16 | Export laporan | `export_summary_report()` |

---

### Perubahan MediaPipe API (penting untuk Python 3.11.9)

| API Lama (dihapus) | API Baru (Tasks, notebook ini) |
|---|---|
| `mp.solutions.face_mesh` | `from mediapipe.tasks.python.vision import FaceLandmarker` |
| `FaceMesh(static_image_mode=True)` | `FaceLandmarker.create_from_options(opts)` |
| `face_mesh.process(rgb)` | `landmarker.detect(mp.Image(...))` |
| `results.multi_face_landmarks` | `result.face_landmarks` |
| Model built-in di package | Butuh `face_landmarker.task` (didownload di Cell 3) |
| Tidak perlu tutup | Wajib `landmarker.close()` setelah selesai |

---

### Struktur folder
```
models/
  face_landmarker.task         <- didownload otomatis di Cell 3
dataset/
  raw_videos/
    apa_spk01_001.mp4
    ...
  processed_frames/
    sample_001/
      frame_0000.jpg           <- 112x112 px, crop bibir
      frame_0001.jpg
      ...
    sample_002/
    ...
  labels.csv
  dataset_stats.png
  REPORT.md
```
